# Grating Coupler Inverse Design

This notebook demonstrates **gradient-based inverse design** of a silicon photonic grating coupler using the Hyperwave SDK.

**Workflow:**
1. Define grating coupler geometry and physical parameters
2. Build initial design (theta) with waveguide and fan-out region
3. Define 8-layer structure recipe for the GC stack
4. Run optimization loop using `compute_adjoint_gradient()` on cloud GPU
5. Visualize results

**Key features:**
- Server-side Gaussian source generation (no large data transfer)
- Two-layer GC design (etched top + unetched bottom silicon)
- Gzip-compressed API requests for fast iteration
- Adjoint method for memory-efficient gradient computation

## Installation

In [ ]:
# Install from GitHub
!pip install -q git+https://github.com/spinsphotonics/hyperwave-community.git

## Configure API

Get your API key from [spinsphotonics.com](https://spinsphotonics.com).

In [ ]:
import hyperwave_community as hwc

# Configure and validate API key
account = hwc.configure_api(
    api_key="YOUR-API-KEY-HERE",
)

hwc.get_account_info();

---
## 1. Physical Parameters

Standard silicon-on-insulator (SOI) grating coupler at 1550 nm:
- **220 nm silicon** device layer with **110 nm partial etch**
- SiO2 buried oxide (BOX) and cladding
- 40-degree fan-out taper angle

In [ ]:
import math
import numpy as np

# Resolution
pixel_size = 0.0175          # um per pixel (17.5 nm design grid)
permittivity_voxel_size = 0.035  # um per voxel (35 nm simulation grid)

# Material refractive indices at 1550 nm
n_silicon = 3.48
n_silicon_dioxide = 1.44
n_chip_cladding = 1.44
n_air = 1.0

# Wavelength
center_wavelength_um = 1.55

# Device geometry (in um)
waveguide_length_um = 5.0
waveguide_width_um = 0.5
h_dev = 0.220        # Device layer thickness (220 nm silicon)
h_box = 2.0          # BOX layer thickness
h_clad = 0.78        # Cladding thickness
h_sub = 0.8          # Substrate thickness
h_air = 1.0          # Air region above
etch_d = 0.110       # Partial etch depth (110 nm)
pad_height = 3.0     # Absorber padding

# Grating coupler taper
gc_taper_angle_deg = 40
taper_length_um = 16

print(f"Resolution: {pixel_size * 1000:.1f} nm/pixel")
print(f"Silicon: n={n_silicon}, eps={n_silicon**2:.2f}")
print(f"SiO2: n={n_silicon_dioxide}, eps={n_silicon_dioxide**2:.2f}")
print(f"Device: {h_dev * 1000:.0f} nm, Etch: {etch_d * 1000:.0f} nm")

---
## 2. Grid and Layer Stack

In [ ]:
# Convert thicknesses to simulation pixels (permittivity grid)
h_p = pad_height / permittivity_voxel_size
h0 = h_air / permittivity_voxel_size
h1 = h_clad / permittivity_voxel_size
h2 = etch_d / permittivity_voxel_size       # Top silicon (etched)
h3 = (h_dev - etch_d) / permittivity_voxel_size  # Bottom silicon (unetched)
h4 = h_box / permittivity_voxel_size
h5 = h_sub / permittivity_voxel_size

# Total Z dimension
Lz = int(math.ceil(h_p + h0 + h1 + h2 + h3 + h4 + h5 + h_p))

# XY grid: 40 um x 40 um domain
grid_size = int(40 / pixel_size)
if grid_size % 2 == 1:
    grid_size -= 1
grid_shape = (grid_size, grid_size)

# Structure dimensions (after 2x downsampling from density filtering)
Lx = grid_size // 2
Ly = grid_size // 2

# Z positions of device layers
design_z_start = int(h_p + h0 + h1)
design_z_end = int(h_p + h0 + h1 + h2 + h3)

print(f"Theta (design) grid: {grid_shape}")
print(f"Structure (simulation) grid: {Lx} x {Ly} x {Lz}")
print(f"Physical size: {grid_size * pixel_size:.1f} x {grid_size * pixel_size:.1f} x {Lz * permittivity_voxel_size:.1f} um")
print(f"\nLayer thicknesses (pixels):")
print(f"  Pad: {h_p:.1f}, Air: {h0:.1f}, Clad: {h1:.1f}")
print(f"  Top Si (etched): {h2:.1f}, Bottom Si: {h3:.1f}")
print(f"  BOX: {h4:.1f}, Substrate: {h5:.1f}")
print(f"  Device Z range: [{design_z_start}, {design_z_end}]")

---
## 3. Build Initial Design (Theta)

The grating coupler has two design layers:
- **Top layer (theta_top)**: Partially etched silicon - this is what we optimize
- **Bottom layer (theta_bottom)**: Solid silicon wherever the waveguide/design region exists

The design includes:
- A fixed waveguide (not optimized)
- A fan-out region defined by the taper angle
- A square optimization region within the fan

In [ ]:
xx, yy = grid_shape

# Coordinate grids
x_coords = np.arange(xx) * pixel_size
y_coords = np.arange(yy) * pixel_size
X, Y = np.meshgrid(x_coords, y_coords, indexing='ij')
y_center = yy * pixel_size / 2
Y_centered = Y - y_center

# Waveguide mask (fixed, not optimized)
waveguide_mask = (X <= waveguide_length_um) & (np.abs(Y_centered) <= waveguide_width_um / 2)

# Fan-out taper geometry
gc_taper_angle_rad = np.deg2rad(gc_taper_angle_deg)
gc_offset = (waveguide_width_um / 2 / np.sin(gc_taper_angle_rad / 2)) * np.cos(gc_taper_angle_rad / 2)
x_focal = waveguide_length_um - gc_offset
r_extension_outer = taper_length_um + 15

fan_region_mask = (
    ((X - x_focal)**2 + Y_centered**2 <= r_extension_outer**2) &
    (np.abs(np.arctan2(Y_centered, X - x_focal)) <= gc_taper_angle_rad / 2) &
    (X >= x_focal) &
    (X > waveguide_length_um)
)

# Absorber widths in simulation pixels
abs_widths = (
    int(2.0 / permittivity_voxel_size),  # X: ~57 pixels
    int(2.0 / permittivity_voxel_size),  # Y: ~57 pixels
    int(3.5 / permittivity_voxel_size),  # Z: ~100 pixels
)

# Square design region (within fan, excluding absorbers)
fan_x_coords = np.where(np.any(fan_region_mask, axis=1))[0]
fan_y_coords = np.where(np.any(fan_region_mask, axis=0))[0]
fan_x_min, fan_x_max = fan_x_coords.min(), fan_x_coords.max()
square_size = max(fan_x_max - fan_x_min, fan_y_coords.max() - fan_y_coords.min())
center_x = (fan_x_min + fan_x_max) // 2
center_y = yy // 2

square_x_start = max(int(waveguide_length_um / pixel_size), center_x - square_size // 2)
square_x_end = min(xx - abs_widths[0] * 2, square_x_start + square_size)
square_y_start = max(abs_widths[1] * 2, center_y - square_size // 2)
square_y_end = min(yy - abs_widths[1] * 2, square_y_start + square_size)

print(f"Waveguide: {waveguide_mask.sum()} pixels")
print(f"Design region: X=[{square_x_start}, {square_x_end}], Y=[{square_y_start}, {square_y_end}]")

In [ ]:
# Initialize theta (two-layer design)
theta_top = np.zeros(grid_shape, dtype=np.float32)
theta_bottom = np.zeros(grid_shape, dtype=np.float32)

# Waveguide is solid silicon in both layers
theta_top[waveguide_mask] = 1.0
theta_bottom[waveguide_mask] = 1.0

# Design region (excludes waveguide)
square_mask = np.zeros(grid_shape, dtype=bool)
square_mask[square_x_start:square_x_end, square_y_start:square_y_end] = True
square_mask = square_mask & ~waveguide_mask

# Top layer: 0.5 (partially etched) in design region
# Bottom layer: solid silicon in design region
theta_top[square_mask] = 0.5
theta_bottom[square_mask] = 1.0

# Design region in structure coordinates (after 2x downsampling)
design_x_start = square_x_start // 2
design_x_end = square_x_end // 2
design_y_start = square_y_start // 2
design_y_end = square_y_end // 2

print(f"Theta top range: [{theta_top.min():.2f}, {theta_top.max():.2f}]")
print(f"Theta bottom range: [{theta_bottom.min():.2f}, {theta_bottom.max():.2f}]")
print(f"Design region (structure): X=[{design_x_start}, {design_x_end}], Y=[{design_y_start}, {design_y_end}]")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Top layer
ax = axes[0]
im = ax.imshow(theta_top.T, aspect='equal', origin='lower', cmap='binary')
ax.set_title('Theta Top (Etched Layer)')
ax.set_xlabel('X (pixels)')
ax.set_ylabel('Y (pixels)')
for val in [square_x_start, square_x_end]:
    ax.axvline(val, color='r', linestyle='--', alpha=0.5)
for val in [square_y_start, square_y_end]:
    ax.axhline(val, color='r', linestyle='--', alpha=0.5)
plt.colorbar(im, ax=ax)

# Bottom layer
ax = axes[1]
im = ax.imshow(theta_bottom.T, aspect='equal', origin='lower', cmap='binary')
ax.set_title('Theta Bottom (Unetched Layer)')
ax.set_xlabel('X (pixels)')
plt.colorbar(im, ax=ax)

# Waveguide mask
ax = axes[2]
im = ax.imshow(waveguide_mask.T.astype(float), aspect='equal', origin='lower', cmap='Blues')
ax.set_title('Waveguide Mask (Fixed)')
ax.set_xlabel('X (pixels)')
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

---
## 4. Structure Spec and Source Configuration

The `layers_template` format defines the 8-layer stack:
1. Top pad (air, absorber region)
2. Air gap above grating
3. Cladding (SiO2)
4. **Top silicon** - etched layer, patterned by `theta_top` (optimized)
5. **Bottom silicon** - unetched layer, patterned by `theta_bottom` (fixed)
6. BOX (buried oxide)
7. Silicon substrate
8. Bottom pad (substrate, absorber region)

The Gaussian source is generated **on the GPU** to avoid transferring large arrays.

In [ ]:
# 8-layer GC structure recipe
layers_template = [
    {'layer_type': 'slab', 'params': {'permittivity': n_air**2, 'thickness': h_p}},
    {'layer_type': 'slab', 'params': {'permittivity': n_air**2, 'thickness': h0}},
    {'layer_type': 'slab', 'params': {'permittivity': n_chip_cladding**2, 'thickness': h1}},
    {'layer_type': 'top', 'params': {'permittivity': (n_chip_cladding**2, n_silicon**2), 'thickness': h2}},
    {'layer_type': 'bottom', 'params': {'permittivity': (n_chip_cladding**2, n_silicon**2), 'thickness': h3}},
    {'layer_type': 'slab', 'params': {'permittivity': n_silicon_dioxide**2, 'thickness': h4}},
    {'layer_type': 'slab', 'params': {'permittivity': n_silicon**2, 'thickness': h5}},
    {'layer_type': 'slab', 'params': {'permittivity': n_silicon**2, 'thickness': h_p}},
]

structure_spec = {
    'grid_shape': list(grid_shape),
    'layers_template': layers_template,
}

print("Layer stack:")
for i, layer in enumerate(layers_template):
    ltype = layer['layer_type']
    p = layer['params']
    thick_um = p['thickness'] * permittivity_voxel_size
    print(f"  {i+1}. {ltype:6s}: eps={str(p['permittivity']):20s}  thickness={p['thickness']:.1f} px ({thick_um:.3f} um)")

In [ ]:
# Frequency band (single wavelength at 1550 nm)
wl_px = center_wavelength_um / permittivity_voxel_size
omega = 2 * np.pi / wl_px
freq_band = (float(omega), float(omega), 1)

# Source: Gaussian beam from above the grating (generated on GPU)
source_pos_z_um = pad_height + h_air - 0.05  # Just above cladding
source_pos_z_px = int(source_pos_z_um / permittivity_voxel_size)
source_offset = (0, 0, source_pos_z_px)

# Gaussian source parameters (generated on GPU, no data transfer needed)
gaussian_source_params = {
    'structure_shape': [Lx, Ly, Lz],
    'source_pos': [0, 0, source_pos_z_px],  # Center of XY, above grating
    'waist_radius': 5.2,  # Standard SMF-28 mode field radius in pixels
    'theta_angle': 0.0,   # Normal incidence
    'phi_angle': 0.0,
    'x_span': 1.0,
    'y_span': 1.0,
    'dz': 0.08,
}

# Absorber parameters
absorption_coeff = 2e-3

# Loss monitor: at waveguide output (maximize forward power Sx)
output_monitor_x = 100  # X position in structure coordinates
loss_monitor_shape = (1, Ly, Lz)
loss_monitor_offset = (output_monitor_x, 0, 0)

# Design monitor: covers full device region for gradient computation
design_monitor_shape = (Lx, Ly, design_z_end - design_z_start)
design_monitor_offset = (0, 0, design_z_start)

print(f"Wavelength: {wl_px:.1f} px ({center_wavelength_um * 1000:.0f} nm)")
print(f"Frequency (omega): {omega:.6f}")
print(f"Source offset: {source_offset}")
print(f"Loss monitor: shape={loss_monitor_shape}, offset={loss_monitor_offset}")
print(f"Design monitor: shape={design_monitor_shape}, offset={design_monitor_offset}")
print(f"Absorber widths: {abs_widths}")

---
## 5. Optimization Settings

In [ ]:
import time

# Optimization hyperparameters
NUM_STEPS = 20           # Optimization steps (increase for better results)
LEARNING_RATE = 0.1      # Adam learning rate
DENSITY_RADIUS = 1       # Density filter radius
DENSITY_ALPHA = 0.8      # Binarization strength (0=none, 1=full)
GPU_TYPE = "B200"        # GPU type (B200, H200, H100, A100, etc.)

print(f"Steps: {NUM_STEPS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Density radius: {DENSITY_RADIUS}")
print(f"Density alpha: {DENSITY_ALPHA}")
print(f"GPU: {GPU_TYPE}")

---
## 6. Run Optimization

Each step:
1. Sends theta to cloud GPU via `compute_adjoint_gradient()`
2. GPU builds 3D structure from theta, runs forward + adjoint FDTD
3. Returns gradient and loss value
4. Local Adam update on theta

In [ ]:
# Adam optimizer state
theta_current = theta_top.copy()
m = np.zeros_like(theta_current)  # First moment
v = np.zeros_like(theta_current)  # Second moment
beta1, beta2, eps_adam = 0.9, 0.999, 1e-8

# Waveguide mask for fixing waveguide region
wg_mask = waveguide_mask.astype(np.float32)
design_mask = square_mask.astype(np.float32)

# History tracking
loss_history = []
theta_history = []

print("=" * 60)
print("GRATING COUPLER INVERSE DESIGN")
print("=" * 60)
print(f"Grid: {Lx} x {Ly} x {Lz}, Theta: {grid_shape}")
print(f"Objective: Maximize forward power (Sx) at waveguide output")
print()

t_start = time.time()

for step in range(1, NUM_STEPS + 1):
    step_start = time.time()

    # Call cloud GPU for adjoint gradient
    result = hwc.compute_adjoint_gradient(
        theta=theta_current,
        theta_bottom=theta_bottom,
        gaussian_source_params=gaussian_source_params,
        source_offset=source_offset,
        freq_band=freq_band,
        loss_monitor_shape=loss_monitor_shape,
        loss_monitor_offset=loss_monitor_offset,
        design_monitor_shape=design_monitor_shape,
        design_monitor_offset=design_monitor_offset,
        structure_spec=structure_spec,
        power_axis=0,             # Maximize Sx (power into waveguide)
        power_maximize=True,
        absorption_widths=abs_widths,
        absorption_coeff=absorption_coeff,
        density_radius=DENSITY_RADIUS,
        density_alpha=DENSITY_ALPHA,
        gpu_type=GPU_TYPE,
    )

    if result is None:
        print(f"Step {step}: API error, stopping.")
        break

    loss_val = result['loss']
    grad = result['grad_theta']
    grad_time = result['grad_time']

    # Mask gradient: only update design region, keep waveguide fixed
    grad = grad * design_mask

    # Adam update
    m = beta1 * m + (1 - beta1) * grad
    v = beta2 * v + (1 - beta2) * grad**2
    m_hat = m / (1 - beta1**step)
    v_hat = v / (1 - beta2**step)
    theta_current = theta_current - LEARNING_RATE * m_hat / (np.sqrt(v_hat) + eps_adam)

    # Clip to [0, 1] and restore waveguide
    theta_current = np.clip(theta_current, 0, 1)
    theta_current[waveguide_mask] = 1.0

    loss_history.append(loss_val)
    theta_history.append(theta_current.copy())

    step_time = time.time() - step_start
    print(f"Step {step}/{NUM_STEPS}: loss={loss_val:.6f}, "
          f"grad=[{result['grad_min']:.2e}, {result['grad_max']:.2e}], "
          f"gpu={grad_time:.1f}s, total={step_time:.1f}s")

total_time = time.time() - t_start
print()
print("=" * 60)
print(f"OPTIMIZATION COMPLETE")
print(f"Total time: {total_time:.1f}s ({total_time/60:.1f} minutes)")
print(f"Best loss: {min(loss_history):.6f} at step {np.argmin(loss_history) + 1}")
print("=" * 60)

---
## 7. Visualize Results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Loss curve
ax = axes[0, 0]
steps = np.arange(1, len(loss_history) + 1)
ax.plot(steps, loss_history, 'b-o', linewidth=2, markersize=5)
ax.set_xlabel('Step')
ax.set_ylabel('Loss (negative power)')
ax.set_title('Optimization Progress')
ax.grid(True, alpha=0.3)

# Initial design
ax = axes[0, 1]
im = ax.imshow(theta_top.T, aspect='equal', origin='lower', cmap='binary', vmin=0, vmax=1)
ax.set_title('Initial Design (theta_top)')
ax.set_xlabel('X (pixels)')
ax.set_ylabel('Y (pixels)')
plt.colorbar(im, ax=ax)

# Final design
ax = axes[1, 0]
if len(theta_history) > 0:
    final_theta = theta_history[-1]
    im = ax.imshow(final_theta.T, aspect='equal', origin='lower', cmap='binary', vmin=0, vmax=1)
    ax.set_title(f'Final Design (Step {len(theta_history)})')
    plt.colorbar(im, ax=ax)
else:
    ax.text(0.5, 0.5, 'No results', ha='center', va='center', transform=ax.transAxes)
ax.set_xlabel('X (pixels)')
ax.set_ylabel('Y (pixels)')

# Best design
ax = axes[1, 1]
if len(theta_history) > 0:
    best_idx = np.argmin(loss_history)
    best_theta = theta_history[best_idx]
    im = ax.imshow(best_theta.T, aspect='equal', origin='lower', cmap='binary', vmin=0, vmax=1)
    ax.set_title(f'Best Design (Step {best_idx + 1}, loss={loss_history[best_idx]:.4f})')
    plt.colorbar(im, ax=ax)
else:
    ax.text(0.5, 0.5, 'No results', ha='center', va='center', transform=ax.transAxes)
ax.set_xlabel('X (pixels)')
ax.set_ylabel('Y (pixels)')

plt.tight_layout()
plt.show()

In [ ]:
# Zoomed view of design region
if len(theta_history) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Initial (zoomed)
    ax = axes[0]
    initial_zoom = theta_top[square_x_start:square_x_end, square_y_start:square_y_end]
    im = ax.imshow(initial_zoom.T, aspect='equal', origin='lower', cmap='binary', vmin=0, vmax=1)
    ax.set_title('Initial Design Region')
    ax.set_xlabel('X (relative)')
    ax.set_ylabel('Y (relative)')
    plt.colorbar(im, ax=ax)

    # Final (zoomed)
    ax = axes[1]
    best_idx = np.argmin(loss_history)
    final_zoom = theta_history[best_idx][square_x_start:square_x_end, square_y_start:square_y_end]
    im = ax.imshow(final_zoom.T, aspect='equal', origin='lower', cmap='binary', vmin=0, vmax=1)
    ax.set_title(f'Best Design Region (loss={loss_history[best_idx]:.4f})')
    ax.set_xlabel('X (relative)')
    ax.set_ylabel('Y (relative)')
    plt.colorbar(im, ax=ax)

    plt.tight_layout()
    plt.show()
else:
    print("No optimization results to display.")

---
## Summary

| Parameter | Value | Description |
|-----------|-------|-------------|
| Wavelength | 1550 nm | Telecom C-band |
| Resolution | 17.5 nm/pixel | High-res for grating features |
| Taper angle | 40 degrees | Fan-out angle |
| Device layer | 220 nm Si | Standard SOI |
| Etch depth | 110 nm | Partial etch |
| Loss function | Power (Sx) | Maximize forward coupling |
| Source | Gaussian (GPU) | Generated on cloud GPU |

### Tips for Better Results
- Increase `NUM_STEPS` to 50-100 for higher efficiency
- Reduce `DENSITY_ALPHA` to 0.0 for freeform optimization, increase later for binarization
- Use `mode_field` + `mode_coupling_params` instead of `power_axis` for mode-matched coupling
- Save checkpoints by writing `theta_history` to disk for resumable runs